In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("OPENAI_API_KEY is not set")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## PART 1 — Tools in AI Agents

**Task 1: Understanding Tools (Conceptual)**
Answer briefly:

1. What is a tool in an AI Agent? → a callable function the model can use (search, calculator, DB, datetime…). it has a name, description, and args schema so the LLM knows when/how to call it.
2. Why do agents need tools? → LLMs alone cant reliably do live search, exact math, or hit your APIs. tools give them actions + fresh/grounded data.
3. Difference between a chatbot and an agent → chatbot mostly replies from prompt/context. an agent can *decide* to call tools, observe results, and loop until it can answer.


**Task 2: Built-in Tools in LangChain**
Use at least 3 built-in tools, such as:

- Calculator
- Wikipedia
- Web search (SerpAPI / Tavily)

Steps:

1. Initialize tools.
2. Test each tool individually.
3. Print tool outputs.


In [5]:

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_tavily import TavilySearch


In [6]:
wikipedia_api_wrapper = WikipediaAPIWrapper()
tavily_search = TavilySearch()

In [7]:
print("Tavily Search:")
print(tavily_search.run("What is the capital of France?"))
print("Wikipedia:")
print(wikipedia_api_wrapper.run("What is the capital of France?"))

Tavily Search:
{'query': 'What is the capital of France?', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://brainly.in/question/62118432', 'title': 'What is the capital city of France and why is it famous?', 'content': 'Answer:\n\nThe capital city of France is Paris.\n\nIt’s famous for several reasons:\n\nCultural heritage – Paris is home to iconic landmarks like the Eiffel Tower, Louvre Museum (housing the Mona Lisa), and Notre-Dame Cathedral.\n\nFashion capital – Known as the global hub for fashion and luxury brands, hosting events like Paris Fashion Week.\n\nHistory – The city has played a major role in European history, from the French Revolution to modern politics. [...] meham495   meham495\n\n Geography\n Secondary School\n\n# What is the capital city of France and why is it famous?\n\nkhanmoominah   khanmoominah\n\nAnswer:\n\nParis is the capital of France, famous for its history, culture, and beauty. Known as the “City of Light,” it has la

In [8]:
from langchain_core.tools import tool
from typing import Literal

@tool
def calculator(x: float, y: float, operation: Literal["add", "subtract", "multiply", "divide"]) -> float:
    """
    Perform a calculation on two numbers.
    
    Args:
        x (float): The first number.
        y (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]): The operation to perform.
        
    Returns:
        float: The result of the calculation.
    """
    if operation == "add":
        return x + y
    elif operation == "subtract":
        return x - y
    elif operation == "multiply":
        return x * y
    elif operation == "divide":
        return x / y
    else:
        raise ValueError("Invalid operation")

In [9]:
calculator.invoke({"x": 2, "y": 2, "operation": "add"})

4.0

## PART 2 — Creating Custom Tools & Toolkits

**Task 3: Create a Custom Tool**
Create a custom tool:

```
@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information"""
```

1. Implement logic (mock data is fine).
2. Test the tool independently.

In [11]:
mock_policy_data = {
    "12345": "This is a mock policy",
    "67890": "This is another mock policy"
}

@tool
def company_policy_lookup(query: str) -> str:
    """Returns company policy information

    Args:
        query (str): The query to search for.
        
    Returns:
        str: The company policy information.
    """
    return mock_policy_data.get(query, "Policy not found")

In [12]:
company_policy_lookup.invoke({"query": "12345"})

'This is a mock policy'

**Task 4: Create a Custom Toolkit**

1. Group multiple tools into a Toolkit.
2. Example tools:
   - Policy lookup
   - Simple database query
   - Date/time tool

In [19]:
from datetime import datetime

In [20]:
@tool
def date_time_tool() -> str:
    """Returns the current date and time"""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


In [22]:
date_time_tool.invoke({})

'2026-09-06 16:34:43'

In [26]:
toolkit = [company_policy_lookup, calculator, tavily_search, date_time_tool]

toolkit

[StructuredTool(name='company_policy_lookup', description='Returns company policy information\n\n    Args:\n        query (str): The query to search for.\n\n    Returns:\n        str: The company policy information.', args_schema=<class 'langchain_core.utils.pydantic.company_policy_lookup'>, func=<function company_policy_lookup at 0x10e5efc40>),
 StructuredTool(name='calculator', description='Perform a calculation on two numbers.\n\nArgs:\n    x (float): The first number.\n    y (float): The second number.\n    operation (Literal["add", "subtract", "multiply", "divide"]): The operation to perform.\n\nReturns:\n    float: The result of the calculation.', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x10e53cd60>),
 TavilySearch(api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None)),
 StructuredTool(name='date_time_tool', description='Returns the current date and time', args_schema=<class 'langchain_co

## PART 3 — Tool Binding & Tool Calling

**Task 5: Tool Binding to LLM**

1. Bind tools to an LLM using LangChain.
2. Allow the LLM to decide when to call which tool.

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [28]:
llm_with_tools = llm.bind_tools(toolkit)

response = llm_with_tools.invoke("What is the capital of France?")
print(response)

response = llm_with_tools.invoke("What is the current date and time?")
print(response)



content='The capital of France is Paris.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 1343, 'total_tokens': 1351, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_335dacabb2', 'id': 'chatcmpl-EL520enbZTuYBFLqWrQ1ddsUI1i6l', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a07668-2f66-7b53-b2d5-7a7d7edb9949-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1343, 'output_tokens': 8, 'total_tokens': 1351, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
content='' addition

In [29]:
response.tool_calls

[{'name': 'date_time_tool',
  'args': {},
  'id': 'call_AjEkG2i1cgnzGj1UZQGZwxLj',
  'type': 'tool_call'}]

**Task 6 — Tool Calling Flow**
Demonstrate:
User Query → LLM → Tool Selection → Tool Execution → Final Answer

Test with queries that require:

- Single tool
- Multiple tools


In [30]:
qsn = [
    "What is current date and time?",
    "what is 2 + (3*9)"
]

In [33]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

In [32]:
tool_map = { t.name: t for t in toolkit}

In [36]:
for q in qsn:
    print(f"Query: {q}")
    messages = [HumanMessage(content=q)]
    response = llm_with_tools.invoke(messages)
    print(response)
    messages.append(response)
    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        tool_input = tool_call["args"]
        tool = tool_map[tool_name]
        tool_response = tool.invoke(tool_input)
        print(tool_response)
        messages.append(ToolMessage(content=str(tool_response), tool_call_id=tool_call["id"]))
    final_response = llm_with_tools.invoke(messages)
    messages.append(AIMessage(content=str(final_response)))
    print(f"Final Response: {final_response}")
    print("-"*100)


Query: What is current date and time?
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 1343, 'total_tokens': 1354, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1280, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_335dacabb2', 'id': 'chatcmpl-EL59PpVyIWVbmWzwmtVvku87yXbQE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0766f-342d-7502-a273-729dd529f5c8-0' tool_calls=[{'name': 'date_time_tool', 'args': {}, 'id': 'call_nWiukBsIpn6rLtSVgjgSRHlX', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 1343, 'output_tokens': 11, 'total_tokens': 1354, 'input_toke

## PART 4 — Creating a ReAct AI Agent

**Task 7: ReAct Agent Overview (Conceptual)**
Explain:

1. What is ReAct (Reason + Act)? → the model alternates thinking (reason) and taking actions (tool calls), then observes tool results before the next step / final answer.
2. Why ReAct agents are powerful → they can break problems into steps (search → calc → answer), recover from missing info, and use tools instead of guessing. better than one-shot answers for multi-step tasks.


**Task 8: Build a ReAct Agent**

1. Create a ReAct-style prompt.
2. Attach tools to the agent.
3. Allow the agent to:
   - Think
   - Act (call tool)
   - Observe
   - Answer

In [38]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.agents import create_agent

In [47]:
system_prompt = """You are a helpful ReAct-style agent.
Use tools when they help answer the question (search, calculator, date/time, policy lookup).
If you already know the answer and no tool is needed, answer directly.
Be concise.
"""


In [48]:
tools = [company_policy_lookup, calculator, tavily_search, date_time_tool]

agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
    debug=True,
)


In [ ]:
result = agent.invoke({
    "messages": [
        ("user", "What is the capital of France?"),
    ]
})

print(result["messages"][-1].content)


[values] {'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='ddb1e350-d060-44f6-90da-3678bbc4a707')]}
[updates] {'model': {'messages': [AIMessage(content='The capital of France is Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 1389, 'total_tokens': 1397, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EL5McYSXvzkhrzEKz8DcKYx5jFU2V', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0767b-b69c-7742-8a1c-4fc97b9b2d5a-0', tool_calls=[], invalid_tool_

**Task 9: Testing the ReAct Agent**
Test the agent with:

1. Factual question (Wikipedia)
2. Calculation-based question
3. Multi-step reasoning question

Observe the reasoning trace.


In [ ]:
qsns = [
    "What is the current date and time?",
    "what is 2 + (3*9)",
]

for q in qsns:
    result = agent.invoke({"messages": [("user", q)]})
    print(f"Q: {q}")
    print(f"A: {result['messages'][-1].content}")
    print("-" * 60)


[values] {'messages': [HumanMessage(content='What is the current date and time?', additional_kwargs={}, response_metadata={}, id='25c65668-dcca-4241-ae62-59a16be1e45c')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 1390, 'total_tokens': 1401, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 1280, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EL5Mk35MAZ7W5npils613wf06oP4h', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0767b-d589-78b1-b65f-1eacb0c65d1e-0', tool_calls=[{'name': 'date_time_tool', 'args'

**Task 10: Agent Use Case**
Build an agent that can:

- Answer questions
- Perform calculations
- Retrieve information

Agent should automatically choose tools.

In [51]:
tools = [company_policy_lookup, calculator, tavily_search]

agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
    debug=True,
)


In [52]:
qsn = [
    "What are the total population of the capital of Russia?",
    "What is (2+3)*9+7?",
]

for q in qsn:
    result = agent.invoke({"messages": [("user", q)]})
    print(f"Question: {q}")
    print(f"Answer: {result['messages'][-1].content}")
    print("-" * 60)


[values] {'messages': [HumanMessage(content='What are the total population of the capital of Russia?', additional_kwargs={}, response_metadata={}, id='7bc4894a-6332-427c-bf1a-8a3511311419')]}
[updates] {'model': {'messages': [AIMessage(content='The capital of Russia, Moscow, has a population of approximately 12.5 million people as of 2023.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 1377, 'total_tokens': 1402, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EL5QR67ZcVNFdy2aH8xulyF1sly5s', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs

## PART 5 — Mini Project: AI Agent Assistant

**Task 11: Observations & Insights**
Write short answers:

1. Benefits of tool-augmented agents → live/accurate data (search), exact math, custom business logic (policy lookup). less hallucination on things tools can verify.
2. Challenges with agents → wrong tool choice, extra latency/cost from tool loops, brittle prompts, debugging traces is messy, need good tool descriptions.
3. Difference between chains and agents → chains are fixed pipelines (A → B → C). agents choose the path dynamically based on the question (maybe search, maybe calculator, maybe both).
4. When to use agents over RAG → use RAG when answers live in *your docs*. use agents when you need actions/tools (calc, APIs, web, multi-step). often combine both: RAG as a tool inside an agent.


In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain